In [4]:

import pandas as pd
from io import StringIO
import json
import dash
from dash import html, dcc, dash_table
import plotly.express as px
from dash.dependencies import Input, Output
from itables import show

In [ ]:
price_cols = ["bid_price_1", "bid_price_2", "bid_price_3", "ask_price_1", "ask_price_2", "ask_price_3"]
volume_cols = ["bid_volume_1", "bid_volume_2", "bid_volume_3", "ask_volume_1", "ask_volume_2", "ask_volume_3"]

def outlier_remover(row):
    if pd.notna(row["bid_price_3"]):
        row["maker_bid_price_1"], row["maker_bid_volume_1"] = row["bid_price_2"], row["bid_volume_2"]
        row["maker_bid_price_2"], row["maker_bid_volume_2"] = row["bid_price_3"], row["bid_volume_3"]
    else:
        row["maker_bid_price_1"], row["maker_bid_volume_1"] = row["bid_price_1"], row["bid_volume_1"]
        row["maker_bid_price_2"], row["maker_bid_volume_2"] = row["bid_price_2"], row["bid_volume_2"]
    if pd.notna(row["ask_price_3"]):
        row["maker_ask_price_1"], row["maker_ask_volume_1"] = row["ask_price_2"], row["ask_volume_2"]
        row["maker_ask_price_2"], row["maker_ask_volume_2"] = row["ask_price_3"], row["ask_volume_3"]
    else:
        row["maker_ask_price_1"], row["maker_ask_volume_1"] = row["ask_price_1"], row["ask_volume_1"]
        row["maker_ask_price_2"], row["maker_ask_volume_2"] = row["ask_price_2"], row["ask_volume_2"]
        

    return row
#If using log data use this, make sure to change filename here and add file to current folder

with open("27173.log") as file:
    json_data = json.load(file)
prices = pd.read_csv("prices_round_0_day_-1.csv", sep=';')
trades = pd.read_csv("trades_round_0_day_-1.csv", sep=';')

#If using data from datapacket off website use this
#prices = pd.read_csv(StringIO(json_data["activitiesLog"]), sep=';')
#trades = pd.DataFrame(json_data["tradeHistory"])

trades["our_direction"]=trades.apply(lambda row: "buy" if row["buyer"]=="SUBMISSION" else "sell" if row["seller"]=="SUBMISSION" else "bot_trade", axis=1) 
trades["buyer"] = trades["buyer"].fillna("")
trades["seller"] = trades["seller"].fillna("")

eme_prices = prices[prices["product"]=="EMERALDS"]
tom_prices = prices[prices["product"]=="TOMATOES"]

eme_trades = trades[trades["symbol"]=="EMERALDS"]
tom_trades = trades[trades["symbol"]=="TOMATOES"]

#fair value line, calculated by 
tom_prices=tom_prices.apply(outlier_remover, axis=1)
eme_prices=eme_prices.apply(outlier_remover, axis=1)
tom_prices["fair_value"]=(tom_prices["maker_bid_price_1"]+tom_prices["maker_bid_price_2"]+tom_prices["maker_ask_price_1"]+tom_prices["maker_ask_price_2"])/4
eme_prices["fair_value"]=(eme_prices["maker_bid_price_1"]+eme_prices["maker_bid_price_2"]+eme_prices["maker_ask_price_1"]+eme_prices["maker_ask_price_2"])/4

#changes prices data into long format
def longDF(df):
    quotes=[]
    for p_col, v_col in zip(price_cols, volume_cols):
        data = df[["timestamp"]].copy()
        data["price"] = df[p_col].values
        data["volume"] = df[v_col].values
        if "bid" in p_col:
            data["side"] = "bid"
        if "ask" in p_col:
            data["side"] = "ask"
        data["fair_value"]=df["fair_value"].values
        quotes.append(data)
    return pd.concat(quotes, ignore_index=True)

tom_long=longDF(tom_prices)
tom_long.dropna(inplace=True)
eme_long=longDF(eme_prices)
eme_long.dropna(inplace=True)

symbol_map= {"buy": "triangle-up", "sell": "triangle-down", "bot_trade": "star"}
datasets = {"EMERALDS": (eme_long, eme_trades), "TOMATOES": (tom_long, tom_trades)}

#calculate position
dir_map = {"buy": 1, "sell": -1, "bot_trade": 0}
tom_trades["position"]=(tom_trades["quantity"] * tom_trades["our_direction"].map(dir_map)).cumsum()
eme_trades["position"]=(eme_trades["quantity"] * eme_trades["our_direction"].map(dir_map)).cumsum()

#generates scatter plot graphs
def make_figure(ticker):
    prices, trades = datasets[ticker]
    figure=px.scatter(prices, x="timestamp", y="price", size="volume", color="side")
    figure.add_scatter(x=trades["timestamp"], y=trades["price"], mode="markers", name="trades",
                       customdata=trades[["timestamp","quantity", "buyer", "seller", "position"]].values,
                       hovertemplate="timestamp=%{x}<br>price=%{y}<br>volume=%{customdata[1]}<br>buyer=%{customdata[2]}<br>seller=%{customdata[3]}<br>our_position=%{customdata[4]}<extra></extra>",
                       marker_symbol=trades["our_direction"].map(symbol_map), marker_size=20, marker_color="black")
    figure.add_scatter(x=prices["timestamp"], y=prices["fair_value"], mode="markers", name="fair value")
    return figure



#generates interactive tool
def grapher():
    app = dash.Dash(__name__)
    
    app.layout = html.Div([
        dcc.Dropdown(id="ticker_dropdown", options=[{"label": x, "value": x} for x in datasets.keys()],
                     value="EMERALDS"),
        dcc.Graph(id="main_graph"),
    ])
    @app.callback(
        Output("main_graph", "figure"),
        Input("ticker_dropdown", "value"))
    def update_graph(selected_ticker):
        return make_figure(selected_ticker)
    
    app.run(debug=True)
grapher()

In [89]:

tom_trades.head(60)

,timestamp,buyer,seller,symbol,currency,price,quantity,our_direction,position
0,2900,SUBMISSION,,TOMATOES,XIRECS,4998.0,3,buy,3
1,3300,SUBMISSION,,TOMATOES,XIRECS,4997.0,3,buy,6
2,3900,,SUBMISSION,TOMATOES,XIRECS,5002.0,6,sell,0
3,4200,SUBMISSION,,TOMATOES,XIRECS,5001.0,7,buy,7
7,10200,,SUBMISSION,TOMATOES,XIRECS,5014.0,2,sell,5
9,12900,SUBMISSION,,TOMATOES,XIRECS,5003.0,11,buy,16
10,13200,,SUBMISSION,TOMATOES,XIRECS,5010.0,5,sell,11
11,14700,,SUBMISSION,TOMATOES,XIRECS,5009.0,3,sell,8
12,14800,SUBMISSION,,TOMATOES,XIRECS,4997.0,5,buy,13
13,15100,SUBMISSION,,TOMATOES,XIRECS,5002.0,7,buy,20


In [51]:
price_cols = ["bid_price_1", "bid_price_2", "bid_price_3", "ask_price_1", "ask_price_2", "ask_price_3"]
volume_cols = ["bid_volume_1", "bid_volume_2", "bid_volume_3", "ask_volume_1", "ask_volume_2", "ask_volume_3"]


#If using log data use this, make sure to change filename here and add file to current folder

with open("19174.log") as file:
    json_data = json.load(file)
#prices = pd.read_csv("prices_round_0_day_-1.csv", sep=';')
#trades = pd.read_csv("trades_round_0_day_-1.csv", sep=';')

#If using data from datapacket off website use this
prices = pd.read_csv(StringIO(json_data["activitiesLog"]), sep=';')
trades = pd.DataFrame(json_data["tradeHistory"])

trades["our_direction"]=trades.apply(lambda row: "buy" if row["buyer"]=="SUBMISSION" else "sell" if row["seller"]=="SUBMISSION" else "bot_trade", axis=1) 
trades["buyer"] = trades["buyer"].fillna("")
trades["seller"] = trades["seller"].fillna("")

eme_prices = prices[prices["product"]=="EMERALDS"]
tom_prices = prices[prices["product"]=="TOMATOES"]

eme_trades = trades[trades["symbol"]=="EMERALDS"]
tom_trades = trades[trades["symbol"]=="TOMATOES"]
tom_prices["profit_and_loss"]+=5013
tom_prices=tom_prices.apply(outlier_remover, axis=1)
tom_prices["fair_value_diff_1"]=(tom_prices["profit_and_loss"]-(tom_prices["maker_bid_price_1"]*tom_prices["maker_bid_volume_1"]+tom_prices["maker_bid_price_2"]*tom_prices["maker_bid_volume_2"]+tom_prices["maker_ask_price_1"]*tom_prices["maker_ask_volume_1"]+tom_prices["maker_ask_price_2"]*tom_prices["maker_ask_volume_2"])/(tom_prices["maker_bid_volume_1"]+tom_prices["maker_bid_volume_2"]+tom_prices["maker_ask_volume_1"]+tom_prices["maker_ask_volume_2"]))**2
tom_prices["fair_value_diff_2"]=(tom_prices["profit_and_loss"]-(tom_prices["maker_bid_price_1"]+tom_prices["maker_bid_price_2"]+tom_prices["maker_ask_price_1"]+tom_prices["maker_ask_price_2"])/4)**2

print(tom_prices["fair_value_diff_1"].sum(), tom_prices["fair_value_diff_2"].sum())
tom_prices.head(50)

C:\Users\malac\AppData\Local\Temp\ipykernel_23140\1744533167.py:25: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



126.05290880508366 85.93336319923401


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,...,maker_bid_price_1,maker_bid_volume_1,maker_bid_price_2,maker_bid_volume_2,maker_ask_price_1,maker_ask_volume_1,maker_ask_price_2,maker_ask_volume_2,fair_value_diff_1,fair_value_diff_2
0,-1,0,TOMATOES,4999,6,4998,19,NaN,NaN,5013,...,4999,6,4998.0,19.0,5013,6,5014.0,19.0,49.000000,49.000000
3,-1,100,TOMATOES,5000,5,4998,23,NaN,NaN,5013,...,5000,5,4998.0,23.0,5013,5,5014.0,23.0,0.047669,0.003320
5,-1,200,TOMATOES,5000,10,4999,20,NaN,NaN,5013,...,5000,10,4999.0,20.0,5013,10,5015.0,20.0,0.024211,0.005222
6,-1,300,TOMATOES,5000,9,4999,24,NaN,NaN,5014,...,5000,9,4999.0,24.0,5014,9,5015.0,24.0,0.038914,0.038914
8,-1,400,TOMATOES,5000,6,4999,25,NaN,NaN,5014,...,5000,6,4999.0,25.0,5014,6,5015.0,25.0,0.059366,0.059366
11,-1,500,TOMATOES,5000,6,4999,25,NaN,NaN,5014,...,5000,6,4999.0,25.0,5014,6,5015.0,25.0,0.012944,0.012944
13,-1,600,TOMATOES,5000,9,4999,17,NaN,NaN,5014,...,5000,9,4999.0,17.0,5014,9,5015.0,17.0,0.037956,0.037956
14,-1,700,TOMATOES,5000,5,4999,24,NaN,NaN,5014,...,5000,5,4999.0,24.0,5014,5,5015.0,24.0,0.026438,0.026438
16,-1,800,TOMATOES,5000,5,4998,25,NaN,NaN,5013,...,5000,5,4998.0,25.0,5013,5,5014.0,25.0,0.074324,0.011227
19,-1,900,TOMATOES,5000,9,4998,15,NaN,NaN,5013,...,5000,9,4998.0,15.0,5013,9,5015.0,15.0,0.000086,0.000086


In [7]:
price_cols = ["bid_price_1", "bid_price_2", "bid_price_3", "ask_price_1", "ask_price_2", "ask_price_3"]
volume_cols = ["bid_volume_1", "bid_volume_2", "bid_volume_3", "ask_volume_1", "ask_volume_2", "ask_volume_3"]

def outlier_remover(row):
    if pd.notna(row["bid_price_3"]):
        row["maker_bid_price_1"], row["maker_bid_volume_1"] = row["bid_price_2"], row["bid_volume_2"]
        row["maker_bid_price_2"], row["maker_bid_volume_2"] = row["bid_price_3"], row["bid_volume_3"]
    else:
        row["maker_bid_price_1"], row["maker_bid_volume_1"] = row["bid_price_1"], row["bid_volume_1"]
        row["maker_bid_price_2"], row["maker_bid_volume_2"] = row["bid_price_2"], row["bid_volume_2"]
    if pd.notna(row["ask_price_3"]):
        row["maker_ask_price_1"], row["maker_ask_volume_1"] = row["ask_price_2"], row["ask_volume_2"]
        row["maker_ask_price_2"], row["maker_ask_volume_2"] = row["ask_price_3"], row["ask_volume_3"]
    else:
        row["maker_ask_price_1"], row["maker_ask_volume_1"] = row["ask_price_1"], row["ask_volume_1"]
        row["maker_ask_price_2"], row["maker_ask_volume_2"] = row["ask_price_2"], row["ask_volume_2"]
        

    return row

In [61]:
dir_map = {"buy": 1, "sell": -1, "bot_trade": 0}
tom_trades["position"]=(tom_trades["quantity"] * tom_trades["our_direction"].map(dir_map)).cumsum()
tom_trades.head(60)

C:\Users\malac\AppData\Local\Temp\ipykernel_23140\1394586428.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,timestamp,buyer,seller,symbol,currency,price,quantity,our_direction,position
0,2900,SUBMISSION,,TOMATOES,XIRECS,4998.0,3,buy,3
1,3300,SUBMISSION,,TOMATOES,XIRECS,4997.0,3,buy,6
4,10200,,SUBMISSION,TOMATOES,XIRECS,5014.0,2,sell,4
5,13200,,SUBMISSION,TOMATOES,XIRECS,5010.0,5,sell,-1
6,14700,,SUBMISSION,TOMATOES,XIRECS,5009.0,3,sell,-4
7,14800,SUBMISSION,,TOMATOES,XIRECS,4997.0,5,buy,1
8,17400,,SUBMISSION,TOMATOES,XIRECS,5009.0,5,sell,-4
9,18800,,SUBMISSION,TOMATOES,XIRECS,5005.0,5,sell,-9
11,24600,SUBMISSION,,TOMATOES,XIRECS,4989.0,2,buy,-7
12,27300,,,TOMATOES,XIRECS,4993.0,5,bot_trade,-7
